# ☸️ Kubernetes — Ultra-Elaborate Mental Models

> **Every section answers four questions: WHY this exists, WHAT it is, HOW it works, WHEN to use it.**
> Real-world scenarios, ❌ before / ✅ after code, and *"Where this is seen in frameworks"* callouts.

---

**Topics**
1. The K8s Mental Model — The Desired State Machine
2. Architecture — Control Plane vs Data Plane
3. Core Objects — Pod, Deployment, Service, ConfigMap, Secret
4. Scheduling — How Pods End Up on Nodes
5. Networking — Services, Ingress, DNS
6. Storage — PV, PVC, StorageClass
7. Deployments — Rolling Updates, Blue/Green, Canary
8. Resource Management — Requests, Limits, HPA
9. Security — RBAC, Network Policies, Pod Security
10. Real-World: How Spotify Runs K8s at Scale
11. The K8s Architect's Design Framework

---
## 1 · The K8s Mental Model — The Desired State Machine

### 🧠 Mental Model — *Tell Kubernetes WHAT, Not HOW*

> **Kubernetes is a desired-state reconciliation engine. You declare what you WANT ("3 replicas of nginx:1.25 running"), and Kubernetes continuously works to make reality match that declaration. You never say 'start this pod on that node' — you describe the desired outcome and Kubernetes figures out the how. This is the fundamental shift from imperative to declarative operations.**

**WHY it exists:** Before Kubernetes (pre-2014):
- Teams used Bash scripts + cron to "manage" containers
- If a container crashed, nobody noticed until a user reported it
- Scaling = SSH into a server, docker run more containers manually
- Load balancing = manually update nginx config, reload nginx, hope nothing breaks
- Resource allocation = manual spreadsheet tracking "server X has 4 CPUs, I think Y service uses 2"

**The Kubernetes loop:**

```
┌──────────────────────────────────────────────────────────────────┐
│                  The Reconciliation Loop                          │
│                                                                   │
│   You apply YAML          │  K8s reads desired state             │
│   (desired state)         │  Compares with actual state          │
│                           │  Takes action to reconcile           │
│   DESIRED: 3 replicas     │                                       │
│   ACTUAL:  2 replicas     │  → Start 1 more pod                  │
│   (1 pod crashed)         │                                       │
│                           │  DESIRED: 3 replicas                 │
│                           │  ACTUAL:  3 replicas ✅ (converged)  │
│                           │                                       │
│   Node failure!           │  DESIRED: 3 replicas                 │
│   2 pods gone             │  ACTUAL:  1 replica                  │
│                           │  → Start 2 new pods on healthy nodes │
│                           │  ACTUAL:  3 replicas ✅ (converged)  │
└──────────────────────────────────────────────────────────────────┘
                    This loop runs every few seconds — forever.
```

### The Controller Pattern — The Architecture Behind the Magic

Every Kubernetes behavior is implemented by a **controller** — a control loop:

```python
# Pseudocode for how every K8s controller works
while True:
    desired_state = read_from_api_server()   # what you declared in YAML
    actual_state  = observe_the_cluster()    # what's actually running
    
    if desired_state != actual_state:
        reconcile(desired_state, actual_state)  # take action
    
    sleep(reconcile_interval)
```

**Controllers for different objects:**
- `ReplicaSet controller` → maintains the right number of pod replicas
- `Deployment controller` → manages ReplicaSets for rolling updates
- `Endpoints controller` → keeps Service endpoints current with pod IPs
- `Node controller` → notices when nodes stop reporting and evicts their pods
- `HPA controller` → scales Deployments based on CPU/memory metrics

### 🌍 Real-World: Kubernetes Self-Healing in Production

At Dropbox, a hardware failure took down an entire rack of servers at 2 AM. No engineer woke up. Kubernetes detected the node failures via missed heartbeats, marked the nodes as `NotReady`, and rescheduled all affected pods to healthy nodes — automatically, in under 3 minutes. By the time the on-call engineer checked their phone, the system had already recovered. The engineer's only job was to replace the hardware.

**Without Kubernetes:** On-call engineer scrambles to SSH into servers, manually restart services, update load balancer configs, verify each service, and document what happened — 45-minute incident at 2 AM.

**With Kubernetes:** On-call engineer acknowledges the alert, confirms the system self-healed, creates a ticket for hardware replacement — 5-minute acknowledgment at 2 AM.

---
## 2 · Architecture — Control Plane vs Data Plane

### 🧠 Mental Model — *The Brain vs The Muscles*

> **The Control Plane is the brain — it makes decisions. The Data Plane (worker nodes) is the muscles — it runs the work. The Control Plane tells the Data Plane what to do through the API server. Worker nodes never directly communicate with each other — they only talk to the API server. This separation means you can upgrade the Control Plane without affecting running workloads.**

### Component Architecture

```
┌──────────────── CONTROL PLANE ────────────────────────────────────────┐
│                                                                         │
│  kube-apiserver     ← The single gateway. ALL changes go through here.  │
│       │               RESTful API, validates resources, stores to etcd  │
│       │                                                                  │
│  etcd (distributed KV store) ← THE source of truth. All cluster state.  │
│                                If etcd is lost, cluster is lost.         │
│  kube-scheduler    ← Assigns pods to nodes (bin packing, affinity rules) │
│  kube-controller-manager ← Runs all built-in controllers                 │
│  cloud-controller-manager ← AWS/GCP/Azure-specific logic (LBs, disks)   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
         ↑↓ API calls (HTTPS)
┌──────────────── WORKER NODES (DATA PLANE) ────────────────────────────┐
│  Node 1                   Node 2                   Node 3              │
│  ┌──────────────┐         ┌──────────────┐         ┌──────────────┐   │
│  │  kubelet     │         │  kubelet     │         │  kubelet     │   │
│  │  (node agent)│         │  (node agent)│         │  (node agent)│   │
│  │  Reports to  │         │  Runs pods   │         │  Health check│   │
│  │  API server  │         │              │         │              │   │
│  ├──────────────┤         ├──────────────┤         ├──────────────┤   │
│  │  kube-proxy  │         │  kube-proxy  │         │  kube-proxy  │   │
│  │  (iptables / │         │  (network    │         │  (Service IP │   │
│  │  eBPF rules) │         │  routing)    │         │  routing)    │   │
│  ├──────────────┤         ├──────────────┤         ├──────────────┤   │
│  │  containerd  │         │  containerd  │         │  containerd  │   │
│  │  (runtime)   │         │  (runtime)   │         │  (runtime)   │   │
│  ├──────────────┤         ├──────────────┤         ├──────────────┤   │
│  │ [Pod] [Pod]  │         │ [Pod] [Pod]  │         │ [Pod]        │   │
│  └──────────────┘         └──────────────┘         └──────────────┘   │
└─────────────────────────────────────────────────────────────────────────┘
```

### etcd — The Most Critical Component

```
etcd stores:
  - All Kubernetes object definitions (Deployments, Services, ConfigMaps)
  - Current cluster state
  - Pod scheduling decisions
  - RBAC policies
  - Secrets (encrypted at rest)

If etcd is lost:
  - Existing pods keep running (kubelet doesn't need API server for running pods)
  - NO new scheduling decisions can be made
  - NO healing (failed pods stay failed)
  - NO config changes possible
  - Essentially: cluster is frozen until etcd is restored

Production requirement: etcd must have:
  - Odd number of nodes (3 or 5) for quorum-based consensus
  - Regular backups (velero or etcdctl snapshot)
  - SSD storage (etcd is extremely I/O-sensitive — high latency = leader election failures)
  - Separate nodes from worker nodes (control plane HA)
```

### 🌍 Real-World: Managed K8s vs Self-Managed

| Service | Who manages Control Plane | Tradeoffs |
|---|---|---|
| **EKS (AWS)** | AWS | Pay $0.10/hr for managed control plane. You manage nodes. Upgrade path handled by AWS. |
| **GKE (Google)** | Google | Auto-upgrades available. Autopilot = fully managed including nodes. |
| **AKS (Azure)** | Azure | Free control plane. Strong AD integration. |
| **Self-managed (kubeadm)** | You | Full control, full responsibility. etcd backup, upgrades, HA all on you. |
| **k3s** | You (but simple) | Lightweight for edge/IoT. Single binary. No HA by default. |

**Senior advice:** Use managed K8s (EKS/GKE/AKS) unless you have a specific reason not to. The control plane is the hardest part to get right. Let the cloud provider solve the 3 AM etcd quorum failures.

In [ ]:
"""
Kubernetes Object Model
=======================
Simulates the Kubernetes API object model and the reconciliation loop.
Understanding this helps you read kubectl output and debug cluster state.
"""
from __future__ import annotations
import json
import random
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional
from datetime import datetime

In [ ]:
class PodPhase(Enum):
    PENDING   = "Pending"    # scheduled but not yet running
    RUNNING   = "Running"    # at least one container running
    SUCCEEDED = "Succeeded"  # all containers exited 0
    FAILED    = "Failed"     # at least one container exited non-0
    UNKNOWN   = "Unknown"    # node lost contact


@dataclass
class Pod:
    name: str
    namespace: str
    image: str
    node: Optional[str] = None
    phase: PodPhase = PodPhase.PENDING
    cpu_request: str = "100m"
    memory_request: str = "128Mi"
    labels: Dict[str, str] = field(default_factory=dict)
    ready: bool = False
    restart_count: int = 0
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())


@dataclass
class Node:
    name: str
    cpu_total: int      # in millicores (1000m = 1 CPU)
    memory_total: int   # in Mi
    cpu_used: int = 0
    memory_used: int = 0
    ready: bool = True
    labels: Dict[str, str] = field(default_factory=dict)

    @property
    def cpu_available(self) -> int:
        return self.cpu_total - self.cpu_used

    @property
    def memory_available(self) -> int:
        return self.memory_total - self.memory_used

    def can_fit(self, cpu_m: int, memory_mi: int) -> bool:
        return self.ready and self.cpu_available >= cpu_m and self.memory_available >= memory_mi


@dataclass
class Deployment:
    name: str
    namespace: str
    image: str
    replicas: int
    selector: Dict[str, str]
    cpu_request: str = "100m"
    memory_request: str = "128Mi"


class KubernetesCluster:
    """Minimal Kubernetes API simulation."""

    def __init__(self, nodes: List[Node]):
        self.nodes: Dict[str, Node] = {n.name: n for n in nodes}
        self.pods: Dict[str, Pod] = {}
        self.deployments: Dict[str, Deployment] = {}
        self._events: List[str] = []

    def _log(self, msg: str) -> None:
        ts = datetime.now().strftime("%H:%M:%S")
        entry = f"[{ts}] {msg}"
        self._events.append(entry)
        print(f"  {entry}")

    def _parse_cpu(self, cpu_str: str) -> int:
        """Convert '250m' → 250, '2' → 2000."""
        if cpu_str.endswith("m"):
            return int(cpu_str[:-1])
        return int(cpu_str) * 1000

    def _parse_memory(self, mem_str: str) -> int:
        """Convert '256Mi' → 256, '1Gi' → 1024."""
        if mem_str.endswith("Mi"):
            return int(mem_str[:-2])
        if mem_str.endswith("Gi"):
            return int(mem_str[:-2]) * 1024
        return int(mem_str)

    def schedule_pod(self, pod: Pod) -> Optional[str]:
        """Scheduler: find the best node for this pod (simplified bin-packing)."""
        cpu_m  = self._parse_cpu(pod.cpu_request)
        mem_mi = self._parse_memory(pod.memory_request)

        # Pick node with most available CPU that can fit the pod
        candidates = [
            n for n in self.nodes.values() if n.can_fit(cpu_m, mem_mi)
        ]
        if not candidates:
            return None

        node = max(candidates, key=lambda n: n.cpu_available)
        node.cpu_used    += cpu_m
        node.memory_used += mem_mi
        return node.name

    def apply_deployment(self, dep: Deployment) -> None:
        """kubectl apply -f deployment.yaml"""
        self.deployments[dep.name] = dep
        self._log(f"Deployment '{dep.name}' created (desired: {dep.replicas} replicas)")
        self._reconcile_deployment(dep)

    def _reconcile_deployment(self, dep: Deployment) -> None:
        """The ReplicaSet controller reconciliation loop."""
        running = [p for p in self.pods.values()
                   if all(p.labels.get(k) == v for k, v in dep.selector.items())
                   and p.phase == PodPhase.RUNNING]

        deficit = dep.replicas - len(running)

        if deficit > 0:
            for i in range(deficit):
                pod_name = f"{dep.name}-{random.randint(10000, 99999)}"
                pod = Pod(
                    name=pod_name,
                    namespace=dep.namespace,
                    image=dep.image,
                    cpu_request=dep.cpu_request,
                    memory_request=dep.memory_request,
                    labels={**dep.selector},
                )
                node = self.schedule_pod(pod)
                if node:
                    pod.node = node
                    pod.phase = PodPhase.RUNNING
                    pod.ready = True
                    self.pods[pod_name] = pod
                    self._log(f"  + Pod '{pod_name}' scheduled on {node}")
                else:
                    pod.phase = PodPhase.PENDING
                    self.pods[pod_name] = pod
                    self._log(f"  ⚠ Pod '{pod_name}' Pending — insufficient resources!")

        elif deficit < 0:
            to_remove = running[:-dep.replicas]
            for pod in to_remove:
                self._log(f"  - Pod '{pod.name}' terminated (scale down)")
                del self.pods[pod.name]

    def simulate_node_failure(self, node_name: str) -> None:
        """Node goes down — kubelet stops reporting. Controller detects and reschedules."""
        node = self.nodes[node_name]
        node.ready = False
        self._log(f"NODE FAILURE: {node_name} is NotReady")

        failed_pods = [p for p in self.pods.values() if p.node == node_name]
        self._log(f"  {len(failed_pods)} pod(s) on failed node — evicting...")

        for pod in failed_pods:
            del self.pods[pod.name]

        # Reconcile all deployments
        for dep in self.deployments.values():
            self._reconcile_deployment(dep)

    def status(self) -> None:
        """kubectl get pods -o wide"""
        print("\n  NAME                      READY  STATUS    NODE")
        print("  " + "-"*60)
        for pod in self.pods.values():
            ready = "1/1" if pod.ready else "0/1"
            print(f"  {pod.name:25s}  {ready}   {pod.phase.value:9s} {pod.node or 'None'}")

In [ ]:
# ── Simulate Kubernetes cluster lifecycle ──────────────────────────────
random.seed(42)

cluster = KubernetesCluster(nodes=[
    Node("node-1", cpu_total=4000, memory_total=8192, labels={"zone": "a"}),
    Node("node-2", cpu_total=4000, memory_total=8192, labels={"zone": "b"}),
    Node("node-3", cpu_total=2000, memory_total=4096, labels={"zone": "c"}),
])

payment_dep = Deployment(
    name="payment-service",
    namespace="production",
    image="registry.company.com/payment-service:sha-a3f4b1c",
    replicas=3,
    selector={"app": "payment-service", "env": "prod"},
    cpu_request="250m",
    memory_request="256Mi",
)

print("=== Applying Deployment ===")
cluster.apply_deployment(payment_dep)
cluster.status()

print("\n=== Simulating Node Failure (self-healing demo) ===")
# Which node has our pods?
pod_nodes = {p.node for p in cluster.pods.values()}
failed_node = list(pod_nodes)[0]
cluster.simulate_node_failure(failed_node)
cluster.status()

print("\n✅ Self-healing: pods automatically rescheduled to healthy nodes")

---
## 3 · Core Objects — Pod, Deployment, Service, ConfigMap, Secret

### 🧠 Mental Model — *The Building Blocks and Why They Exist Separately*

> **Each Kubernetes object exists because it solves a different problem. Pods are the unit of execution. Deployments manage pod lifecycle. Services provide stable network identity. ConfigMaps decouple config from code. Secrets store sensitive config. They are composable — each does one thing well.**

### Object Hierarchy

```
Deployment (you manage this)
  └── ReplicaSet (Deployment manages this automatically)
        └── Pod (ReplicaSet manages this)
              ├── Container (your app)
              ├── Container (sidecar: log agent, proxy)
              └── Volume (shared storage between containers)

Service (network identity, independent of pod lifecycle)
  └── Endpoints (auto-populated with pod IPs)
        └── Selector matches pods by label

ConfigMap / Secret (mounted as files or env vars into pods)
Ingress → Service → Pods (external traffic flow)
```

### Production YAML — The Complete Microservice

```yaml
# ✅ PRODUCTION-GRADE: payment-service complete manifest

---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: payment-service
  namespace: production
  labels:
    app: payment-service
    version: "1.4.2"
    team: payments
spec:
  replicas: 3
  
  # ✅ How to identify pods for this deployment
  selector:
    matchLabels:
      app: payment-service
  
  # ✅ Deployment strategy (default: RollingUpdate)
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxUnavailable: 1   # at most 1 pod down during update
      maxSurge: 1         # at most 1 extra pod during update
  
  template:
    metadata:
      labels:
        app: payment-service
        version: "1.4.2"
    spec:
      
      # ✅ Spread pods across nodes and zones
      topologySpreadConstraints:
        - maxSkew: 1
          topologyKey: topology.kubernetes.io/zone
          whenUnsatisfiable: DoNotSchedule
          labelSelector:
            matchLabels:
              app: payment-service
      
      # ✅ Prefer different nodes for resilience
      affinity:
        podAntiAffinity:
          preferredDuringSchedulingIgnoredDuringExecution:
            - weight: 100
              podAffinityTerm:
                labelSelector:
                  matchLabels:
                    app: payment-service
                topologyKey: kubernetes.io/hostname
      
      serviceAccountName: payment-service-sa  # ✅ minimal RBAC identity
      
      containers:
        - name: payment-service
          image: registry.company.com/payment-service:sha-a3f4b1c  # ✅ pinned SHA
          
          ports:
            - containerPort: 8000
              name: http
          
          # ✅ Resources: ALWAYS set both requests AND limits
          resources:
            requests:
              cpu: "250m"      # scheduler uses this for placement
              memory: "256Mi"  # scheduler uses this for placement
            limits:
              cpu: "500m"      # throttled if exceeded (not killed)
              memory: "512Mi" # KILLED (OOMKilled) if exceeded — set carefully!
          
          # ✅ Env from ConfigMap + Secret
          env:
            - name: LOG_LEVEL
              valueFrom:
                configMapKeyRef:
                  name: payment-config
                  key: LOG_LEVEL
            - name: STRIPE_SECRET_KEY
              valueFrom:
                secretKeyRef:
                  name: payment-secrets
                  key: stripe-secret-key
          
          # ✅ Health probes — CRITICAL for zero-downtime deploys
          startupProbe:         # prevents killing slow-starting apps
            httpGet:
              path: /health
              port: 8000
            failureThreshold: 30
            periodSeconds: 10
          
          readinessProbe:       # controls traffic routing
            httpGet:
              path: /ready
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 10
            failureThreshold: 3
          
          livenessProbe:        # controls pod restart
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            periodSeconds: 20
            failureThreshold: 3
          
          # ✅ Security: minimal permissions
          securityContext:
            runAsNonRoot: true
            runAsUser: 1001
            readOnlyRootFilesystem: true
            allowPrivilegeEscalation: false
            capabilities:
              drop: ["ALL"]
          
          # ✅ Graceful shutdown
          lifecycle:
            preStop:
              exec:
                command: ["/bin/sh", "-c", "sleep 5"]
      
      terminationGracePeriodSeconds: 30

---
apiVersion: v1
kind: Service
metadata:
  name: payment-service
  namespace: production
spec:
  selector:
    app: payment-service  # ← matches pods by label
  ports:
    - port: 80
      targetPort: 8000
      name: http
  type: ClusterIP  # internal only; Ingress exposes externally
```

### The Three Health Probes — Getting Them Wrong Is the #1 K8s Mistake

```
startupProbe:    Is the app done starting up?
                 → K8s waits for this BEFORE checking the others
                 → Set high failureThreshold × periodSeconds for slow apps
                 → Without this: liveness probe kills app mid-startup!

readinessProbe:  Is the app ready to receive traffic?
                 → Failing: pod removed from Service endpoints (no traffic)
                 → Use for: DB connection not ready, cache warming in progress
                 → Does NOT restart the pod — just stops traffic

livenessProbe:   Is the app stuck and needs a restart?
                 → Failing: pod is KILLED and restarted
                 → Only use for genuine deadlocks, infinite loops
                 → DO NOT check DB connectivity here — if DB goes down,
                   you don't want ALL pods restarting simultaneously!
```

---
## 4 · Networking — Services, Ingress, DNS

### 🧠 Mental Model — *The Virtual Network Overlay*

> **Every pod gets a unique IP address. Pods are ephemeral — their IPs change when they restart. A Service provides a stable IP + DNS name that load-balances across all healthy pods. Think of it as a virtual IP that never changes, always pointing to the currently-healthy pods behind it.**

### Traffic Flow: External Request to Pod

```
External User
     │
     ▼
DNS: api.company.com → 34.120.45.67 (Load Balancer IP)
     │
     ▼
Cloud Load Balancer (AWS ALB / GCP Load Balancer)
     │
     ▼
Ingress Controller (nginx, traefik, istio-gateway)
  - TLS termination
  - Path-based routing: /api/payments → payment-service
  - Host-based routing: api.company.com → api-service
     │
     ▼
Service: payment-service (ClusterIP: 10.96.45.22)
  - kube-proxy / eBPF translates ClusterIP → one of the pod IPs
  - Load balancing: round-robin (default) or IPVS (more algorithms)
     │
     ▼
Pod: payment-service-7d4f9c-xk2p1 (Pod IP: 10.244.1.15:8000)
```

### Service Types

| Type | When to use | How it works |
|---|---|---|
| **ClusterIP** | Internal services (default) | Stable IP accessible only within cluster |
| **NodePort** | Dev/testing, simple external access | Opens port on every node (30000-32767) |
| **LoadBalancer** | Production external access | Provisions cloud LB (costs $$) |
| **ExternalName** | DNS alias to external service | CNAME in cluster DNS to external hostname |

```
Best practice for production:
  Internal traffic:  ClusterIP + DNS (payment-service.production.svc.cluster.local)
  External traffic:  1 LoadBalancer Service pointing to Ingress Controller
                    Ingress rules route to ClusterIP services
  Cost optimization: Don't create LoadBalancer per service — one ingress controller handles all
```

### Kubernetes DNS — The Internal Service Discovery

```
Every Service gets a DNS name automatically:
  <service-name>.<namespace>.svc.cluster.local

Examples:
  payment-service.production.svc.cluster.local
  redis.infrastructure.svc.cluster.local
  postgres.databases.svc.cluster.local

Within the same namespace, short names work:
  payment-service (resolves to full DNS name via search domains)

Cross-namespace requires full name:
  redis.infrastructure (or full: redis.infrastructure.svc.cluster.local)

This is how microservices discover each other without hardcoded IPs.
```

---
## 5 · Resource Management — Requests, Limits, HPA

### 🧠 Mental Model — *The Bin-Packing Problem*

> **Resource requests and limits are the language you use to tell Kubernetes how much 'space' each pod needs. Without them, the scheduler is blind — it can't make intelligent placement decisions. Set them wrong, and you'll either waste money (over-provisioned) or cause OOMKills and throttling (under-provisioned). Getting them right is the single highest-impact K8s optimization.**

### Requests vs Limits

```
Requests: what the scheduler uses for placement.
  "I need at least 250m CPU and 256Mi memory"
  → Scheduler finds a node with 250m+ CPU and 256Mi+ memory available
  → Pod is GUARANTEED this minimum

Limits: the maximum the container is allowed to use.
  CPU:    If exceeded → container is CPU-throttled (slows down, not killed)
  Memory: If exceeded → container is KILLED (OOMKilled)

  Setting memory limits too low is the #1 cause of mysterious pod restarts.
```

### The Three QoS Classes

```
BestEffort:    No requests OR limits set
               First to be evicted under node pressure
               ❌ Never use in production

Burstable:     Requests < Limits (can burst)
               Evicted after BestEffort under pressure
               ✅ Use for most services

Guaranteed:    Requests == Limits (exactly)
               Last to be evicted (highest priority)
               ✅ Use for latency-sensitive, critical services
               ⚠️  Can't burst — bad for spiky workloads
```

### Horizontal Pod Autoscaler (HPA)

```yaml
# HPA automatically adjusts replicas based on metrics
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: payment-service-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: payment-service
  minReplicas: 3
  maxReplicas: 20
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70  # scale up when avg CPU > 70%
    - type: Resource
      resource:
        name: memory
        target:
          type: Utilization
          averageUtilization: 80
  # Custom metric from Prometheus (requests per second)
    - type: Pods
      pods:
        metric:
          name: http_requests_per_second
        target:
          type: AverageValue
          averageValue: "100"  # scale when > 100 RPS per pod
```

---

## 6 · The K8s Architect's Design Framework

### The 10 Questions Before Any K8s Deployment

```
1. RESOURCE SIZING
   What are the memory/CPU profiles? (profile the app, don't guess)
   Set requests based on p50 usage, limits based on p99 + buffer.

2. REPLICAS
   What's the minimum HA? (2 replicas minimum — never 1 in prod)
   What's the max scale? (HPA maxReplicas must fit the cluster)

3. HEALTH PROBES
   Does the app need a startup probe? (slow startup? YES)
   What URL is 'ready'? (should check upstream deps)
   What URL is 'live'? (should NOT check upstream deps)

4. NETWORKING
   ClusterIP or LoadBalancer? (internal: ClusterIP, external: Ingress)
   What's the Ingress configuration? (TLS, path routing, auth)

5. STORAGE
   Is the app stateless? (yes → no PVs needed)
   If stateful: what StorageClass? (SSD? NFS? managed storage?)
   Backup strategy for PVs?

6. SECURITY
   Which ServiceAccount? (dedicate one per service, no default SA)
   What RBAC does the SA need? (least privilege)
   Network policy: which services can talk to this?
   Pod security: nonRoot, readOnly filesystem, drop ALL capabilities

7. DEPLOYMENT STRATEGY
   RollingUpdate? Blue/Green? Canary? (Argo Rollouts for advanced)
   What are maxUnavailable and maxSurge?
   Rollback procedure and SLA?

8. CONFIG AND SECRETS
   ConfigMap for non-sensitive config
   Secrets for sensitive (or External Secrets Operator + Vault)
   Never mount config directly from Git — goes through ConfigMap

9. OBSERVABILITY
   What metrics does the app expose? (/metrics for Prometheus)
   What are the SLOs? (what do you alert on?)
   Log aggregation: Fluentd/Fluent Bit → Elasticsearch or CloudWatch

10. DISASTER RECOVERY
    What happens if the entire namespace is deleted?
    All manifests are in Git (GitOps) → reapply and recover
    PV data backup with Velero
    Multi-region failover strategy
```